# Part 4: Conditional BTC Allocation Rule / Risk-Budgeted Allocation Diagnostics

This notebook runs the reproducible Python runner for Part 4. It mounts Google Drive, installs the minimal requirements, executes the runner, and previews validation tables and figures. It does not write thesis conclusions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_ROOT)
print(PROJECT_ROOT)

assert (PROJECT_ROOT / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_ROOT / 'experiments' / 'part4_conditional_btc_allocation' / 'run_part4_conditional_btc_allocation.py').exists(), 'Part 4 runner not found.'

## Path configuration

Default Colab paths assume the canonical Drive structure:

- `outputs/part1_btc_macro_state/colab_part1_seed42`
- `outputs/part2_portfolio_risk_budget/colab_part2_seed42`
- `outputs/part3_btc_state_dependence/colab_part3_seed42`

If testing from locally downloaded zip folders, use these alternatives in the runner command:

- `outputs/part1_btc_macro_state_outputs/part1_btc_macro_state/colab_part1_seed42`
- `outputs/part2_portfolio_risk_budget_outputs/part2_portfolio_risk_budget/colab_part2_seed42`
- `outputs/part3_btc_state_dependence_outputs/part3_btc_state_dependence/colab_part3_seed42`

In [ ]:
PART1_RUN_DIR = Path('outputs/part1_btc_macro_state/colab_part1_seed42')
PART2_RUN_DIR = Path('outputs/part2_portfolio_risk_budget/colab_part2_seed42')
PART3_RUN_DIR = Path('outputs/part3_btc_state_dependence/colab_part3_seed42')
RUN_ID = 'colab_part4_seed42'
SEED = 42

for label, path in [('Part 1', PART1_RUN_DIR), ('Part 2', PART2_RUN_DIR), ('Part 3', PART3_RUN_DIR)]:
    print(label, path, path.exists())
    assert path.exists(), f'{label} run folder not found. Check the path configuration cell.'

In [ ]:
!pip install -q -r experiments/part4_conditional_btc_allocation/requirements-part4.txt

In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    'experiments/part4_conditional_btc_allocation/run_part4_conditional_btc_allocation.py',
    '--input-dir', 'data_2026/cleaned',
    '--part1-run-dir', str(PART1_RUN_DIR),
    '--part2-run-dir', str(PART2_RUN_DIR),
    '--part3-run-dir', str(PART3_RUN_DIR),
    '--output-dir', 'outputs/part4_conditional_btc_allocation',
    '--run-id', RUN_ID,
    '--seed', str(SEED),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd

RUN_DIR = PROJECT_ROOT / 'outputs' / 'part4_conditional_btc_allocation' / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

print(RUN_DIR)
print('Results:', sorted(p.name for p in RESULTS.iterdir()))
print('Figures:', sorted(p.name for p in FIGURES.iterdir()))

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
pd.read_csv(RESULTS / 'allocation_rule_definition.csv')

In [ ]:
pd.read_csv(RESULTS / 'risk_budget_cap_audit.csv')

In [ ]:
pd.read_csv(RESULTS / 'conditional_portfolio_performance_summary.csv')

In [ ]:
pd.read_csv(RESULTS / 'sensitivity_state2_low_allocation_summary.csv')

In [ ]:
from IPython.display import Image, display

for name in [
    'conditional_rule_timeline.png',
    'conditional_portfolio_drawdowns.png',
    'conditional_vs_fixed_btc_summary.png',
    'risk_budget_cap_audit.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
# Optional resume test after a successful run.
# cmd_resume = cmd + ['--resume']
# subprocess.run(cmd_resume, check=True)

In [ ]:
# Optional archive for download from Colab.
import shutil
archive_path = shutil.make_archive(str(RUN_DIR), 'zip', RUN_DIR)
print(archive_path)